In [ ]:
#imports
from dotenv import load_dotenv
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
# from langchain_openai import OpenAIEmbeddings
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import chromadb

#load in the .env variables if using OpenAI
# load_dotenv()

True

In [2]:
#Read State of the Union Address File
with open("state_of_the_union.txt") as f:
    state_of_union = f.read()

#Initialize Text Splitter
text_splitter = CharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    length_function=len
)

# Create Documents (Chunks) from file
texts = text_splitter.create_documents([state_of_union])

In [3]:
# Get embeddings
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
embeddings = OllamaEmbeddings(model='nomic-embed-text')

# Initialize ChromaDB as Vector Store
client = chromadb.HttpClient(host="localhost", port=8000)
vector_store = Chroma(
    client=client,
    collection_name="test_collection",
    embedding_function=embeddings
)

In [ ]:
# Save Document Chunks to Vector Store
ids = vector_store.add_documents(texts)
db = client.get_collection('test_collection')
print(db.peek())
print(db.count())



{'ids': ['3e3694ad-b394-4c59-ace3-05bf6badacdb', '91390401-586e-47c5-b0e2-8b55b7040764', '0e35383e-8811-46ad-8655-3cb950c4e61b', '13713b58-e1a2-45fb-a29e-dd6ee9d9fc08', '902168fc-0f23-46b6-91c3-48e40a980cd2', '9a521350-c42b-40cf-a222-42f5e93982fa', 'b09462db-c4c3-4015-8852-869b367d5c91', 'b5b59567-fee1-4f2a-af64-ee89984daae1', 'dd0b35a5-5a50-4356-b210-89c31c4d5c59', 'c455f497-e6f4-4bcc-be22-574c9f981aaa'], 'embeddings': array([[ 0.03504862, -0.01563167, -0.18109843, ..., -0.05190314,
        -0.05081977, -0.04157535],
       [ 0.04153883,  0.05513857, -0.19456978, ..., -0.04658961,
        -0.03171461, -0.01531389],
       [-0.02702171,  0.00927824, -0.18364848, ..., -0.0554321 ,
        -0.03319491, -0.071215  ],
       ...,
       [ 0.07594796,  0.07851451, -0.17182851, ..., -0.07149892,
        -0.05802611, -0.02654002],
       [ 0.02922696,  0.03285167, -0.16388538, ..., -0.04680729,
        -0.0288415 , -0.03803434],
       [ 0.0362404 ,  0.02595763, -0.17803991, ..., -0.07654689,

In [8]:
# Query the Vector Store to test
results = vector_store.similarity_search(
    'Who invaded Ukraine?',
    k=2
)

# Validate the output, it should include parts of the speech
# that mentions the NATO creation.
for res in results:
    print(f"* {res.page_content} [{res.metadata}]\n\n")


* And a proud Ukrainian people, who have known 30 years  of independence, have repeatedly shown that they will not tolerate anyone who tries to take their country backwards.  

To all Americans, I will be honest with you, as I’ve always promised. A Russian dictator, invading a foreign country, has costs around the world. [{}]


* It matters. American diplomacy matters. American resolve matters. 

Putin’s latest attack on Ukraine was premeditated and unprovoked. 

He rejected repeated efforts at diplomacy. 

He thought the West and NATO wouldn’t respond. And he thought he could divide us at home. Putin was wrong. We were ready.  Here is what we did.   

We prepared extensively and carefully. 

We spent months building a coalition of other freedom-loving nations from Europe and the Americas to Asia and Africa to confront Putin. [{}]




In [ ]:
# Set Chroma Vector Store as the Retriever 
retriever = vector_store.as_retriever()

# Initialize the LLM instance, using Gemma4
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.3,
    top_p=0.95,
    top_k=64
)

# Document parsing function to String
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [10]:
# Create a function that created a prompt template
def gen_prompt_template():
    return ChatPromptTemplate.from_messages(
        [
            ("system", "You are required to answer only based on the context provided. If you don't know, say you don't know."),
            ("human", """Use the context below to answer the question.
            
            context: {context}

            question: {query}
            
            """)
        ]
    )

In [11]:
# Create the RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "query": RunnablePassthrough()}
    | gen_prompt_template()
    | llm
    | StrOutputParser()
)


In [12]:
# Query the RAG chain
rag_chain.invoke("Who invaded Ukraine?")

'A Russian dictator invaded Ukraine.'

In [13]:
# Query the RAG chain
rag_chain.invoke("What is the purpose of Retrieval Augmented Generation (RAG)?")

"I don't know."

In [14]:
# Query the RAG chain
rag_chain.invoke("What is the purpose of NATO Alliance?")

'The forces are mobilizing to defend NATO Allies in the event that Putin decides to keep moving west, with the purpose of protecting NATO countries, including Poland, Romania, Latvia, Lithuania, and Estonia.'